# SPATIAL INTELLIGENCE — PART 1
## Homework 02

## 1. Import the needed libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [2]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.20) is OLDER than the latest version (0.9.30) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [3]:
renderer = "vscode"

## 4. Utility functions

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == 'face_id':
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan outline (OBJ)

In [28]:
OBJ_PATH = r"C:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\assignment02\plan_5.obj"
objects = Topology.ByOBJPath(OBJ_PATH)
print(f"Imported {len(objects)} objects")

floor_face = None
for obj in objects:
    faces = Topology.Faces(obj)
    if faces:
        floor_face = faces[0]
        break
    wires = Topology.Wires(obj)
    if wires:
        floor_face = Face.ByWire(wires[0])
        if floor_face:
            break

print("Floor face loaded:", floor_face is not None)

b_r = Wire.BoundingRectangle(floor_face)
d_br = Topology.Dictionary(b_r)
xmin   = Dictionary.ValueAtKey(d_br, "xmin")
xmax   = Dictionary.ValueAtKey(d_br, "xmax")
ymin   = Dictionary.ValueAtKey(d_br, "ymin")
ymax   = Dictionary.ValueAtKey(d_br, "ymax")
width  = Dictionary.ValueAtKey(d_br, "width")
length = Dictionary.ValueAtKey(d_br, "length")
print(f"Bounds: x=[{xmin:.1f}, {xmax:.1f}]  y=[{ymin:.1f}, {ymax:.1f}]")
print(f"Size: {width:.1f} x {length:.1f} units")

Imported 1 objects
Floor face loaded: True
Bounds: x=[1.0, 24.3]  y=[0.1, 7.9]
Size: 23.3 x 7.8 units


## 6. Show the floor plan

In [29]:
Topology.Show(floor_face,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 7. Create a grid overlay

In [31]:
GRID_STEP = 1
uRange = list(range(0, int(width)  + GRID_STEP, GRID_STEP))
vRange = list(range(0, int(length) + GRID_STEP, GRID_STEP))
grid = Grid.EdgesByDistances(floor_face, clip=True, uRange=uRange, vRange=vRange)

## 8. Show the floor plan and the grid

In [32]:
Topology.Show(floor_face, grid,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 9. Slice the floor plan with the grid to create a topologic shell

In [33]:
shell = Topology.Slice(floor_face, grid)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)
print(f"Grid cells: {len(faces)}")

Grid cells: 250


## 10. Show the resulting shell

In [34]:
Topology.Show(shell,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 11. Derive navigation and analysis graphs from the shell

In [35]:
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph   = Graph.ByTopology(shell)

## 12. Derive and store the analysis graph vertices

In [36]:
g_verts = Graph.Vertices(analysis_graph)

## 13. Show the analysis graph

In [37]:
Topology.Show(analysis_graph,
              camera=[0, 0, 6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 14. Spatial Intelligence through Graph Analysis
### a. Shortest Path

In [38]:
import time

start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2, 0)
end_vertex   = Vertex.ByCoordinates(xmax-2, ymin+2, 0)
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)

t0 = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
print("Shortest Path Duration:", round(time.time()-t0, 2), "seconds")

t0 = time.time()
straight_path = Wire.Straighten(shortest_path, host=floor_face)
print("Straighten Duration:", round(time.time()-t0, 2), "seconds")
print("Original Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Path Length:", round(Wire.Length(straight_path), 2))

for edge in Topology.Edges(shortest_path):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(["width","color"],[7,"red"]))
for edge in Topology.Edges(straight_path):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(["width","color"],[7,"blue"]))

Shortest Path Duration: 0.15 seconds
Straighten Duration: 0.96 seconds
Original Path Length: 24.84
Straightened Path Length: 22.12


In [39]:
Topology.Show(floor_face, shortest_path, straight_path,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

### b. Closeness Centrality / Integration
* Closeness centrality quantifies how close a node is to all other nodes (reciprocal of the sum of shortest-path distances).
* In space syntax this corresponds to global integration.

In [40]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

In [41]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [42]:
Topology.Show(faces,
              faceColorKey="cc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

### c. Betweenness Centrality / Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [45]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

In [46]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [47]:
Topology.Show(faces,
              faceColorKey="bc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)